In [ ]:
# Import general libraries
import numpy as np
import pandas as pd
import re
import pickle
import warnings
import torch
import itertools
from brian2 import *
from sbi import utils, inference
import matplotlib.pyplot as plt
import os
from pathlib import Path
import getpass
import psutil
import torch
from copy import deepcopy
from dataclasses import fields

In [ ]:
# Import from other files in the project
# from simulator_with_force_long_AHP import SimulationParameters, run_simulation
from simulator_with_force_long_AHP_output_force_of_different_pools_separately import SimulationParameters, run_simulation
from analyzer_with_force import AnalyzesParams, analyze_data
from parameter_set_batch_summary import export_parameter_set_summary_table, summarize_parameter_set_batch, summarize_rate_isi_batch


In [ ]:
### If cython complains, clear cache ###
# from brian2 import clear_cache
# clear_cache('cython')

In [ ]:
# PRIOR AND PRIOR SAMPLING PARAMETERS FOR THE SIMULATION BATCH
batch_name_original = "£££¤¤¤£££_sim_example_for_figure" # "$$$_sim_batches_posterior_predictive_checks_TA_NOGO_sbi_with_all_features"
add_to_same_batch_simulation_folder = True  # If True, add new simulations to an existing batch folder.

# OPTIONAL POSTERIOR-DERIVED BASELINE / MODULATION PRIORS
# Use raw simulator parameter names here (e.g. "tau_size_um"), not SIM_PARAM_* export names.
# Each source owns one disjoint parameter block. Sources are applied in order.
# Posterior-derived parameters override fixed_parameters and free_parameter_bounds_prior.
# Overlap with sample_at_specified_values is a hard error.
USE_POSTERIOR_DERIVED_PRIORS = True
POSTERIOR_PRIOR_SOURCES = [
    {
        "name": "baseline_pass",
        "source_kind": "posterior_samples",
        "posterior_samples_path": Path(r"C:/Users/franc/Documents/_New_project_motor_command_dynamics/VS_code_project_MN_sim_with_force/$_sbi_output_BASELINE_grand_mean__run_0004/posterior_samples.pkl"),
        "observation_label": "grand_mean",
        "derived_parameters": [
            "tau_size_um",
            "common_input_weight_sd",
            "excitatory_input_baseline",
            "lf_target_sd",
            "alpha_target_sd_final",
            "beta_target_sd_final",
            "independent_input_power",
            "independent_input_weight_sd",
        ],
        "random_seed":  12345,
    },
    # {
    #     "name": "cue_pass",
    #     "source_kind": "posterior_samples",
    #     "posterior_samples_path": Path(r"C:\Users\franc\Documents\_New_project_motor_command_dynamics\VS_code_project_MN_sim_with_force\$_sbi_output_GONOGO_V3\SBI_RUN_normalize_to_exp_FEATURES_all_windows\posterior_samples.pkl"),
    #     "observation_label": "TA_NOGO", #"TA_NOGO", "HAND_NOGO", "HAND_GO"
    #     "derived_parameters": [
    #         "lf_trend_slope_nA_per_s",
    #         "alpha_trend_slope_nA_per_s",
    #         "beta_trend_slope_nA_per_s",
    #         "lf_burst_peak_nA",
    #         "alpha_burst_peak_nA",
    #         "beta_burst_peak_nA",
    #     ],
    #     "random_seed": 23456,
    # },
]

# Optional composite posterior-mode parameter set. When enabled, one extra
# parameter set is appended by taking the posterior mode for each source
# in POSTERIOR_PRIOR_SOURCES and writing it to a recognized mode folder.
ADD_POSTERIOR_MODE_PARAMETER_SET = True
POSTERIOR_MODE_PARAMETER_SET_LABEL = None  # None -> auto label from source names and observation labels.
POSTERIOR_MODE_FALLBACK_TO_SAMPLES = True
POSTERIOR_MODE_KNN_NEIGHBORS = 25
POSTERIOR_MODE_MIN_NEIGHBORS = 5

# Start from the simulator defaults, then override the parameters that this notebook
# typically edits explicitly. This keeps the notebook resilient when new simulator
# parameters are added in the Python module.
default_simulation_parameters = SimulationParameters()
fixed_parameters = {
    f.name: getattr(default_simulation_parameters, f.name)
    for f in fields(SimulationParameters)
    if f.init
}
fixed_parameters.update({
    "brian_codegen_target": "cython", # "numpy"

    # DIAGNOSTIC OUTPUT CONTROL
    "output_plots": True,
    "output_plot_summary_input_cst_figure": True,
    "output_plot_input_component_figures": True,
    "output_plot_common_input_weight_figure": False,
    "output_plot_independent_input_weight_figure": False,
    "output_plot_sync_diagnostic_figure": True,
    "output_plot_firing_rate_figure": True,
    "output_plot_spike_trains_figure": True,
    "output_plot_mn_size_figure": True,
    "output_plot_mn_properties_figure": False,
    "output_plot_target_baseline_figure": False,
    "output_plot_common_vs_independent_power_figure": True,
    "output_plot_force_tracking_figure": False,
    "output_plot_force_properties_figures": False,
    "output_plot_force_optimization_figures": False,
    "save_common_input_power_csv": False,
    "save_independent_input_power_csv": False,

    # MINIMAL OUTPUT PROFILE
    "minimal_output": False, # True,
    "minimal_output_turn_off_figure_output": True, # True,
    "minimal_output_save_spike_trains": False,
    "minimal_output_downsample_saved_traces": True,
    "minimal_output_target_fsamp": 100,

    # GENERAL SIMULATION SETTINGS
    "pre_specify_random_seed": False,
    "duration": 6.0,
    "edges_ignore_duration": 2,
    "get_ready_cue_time_s": 4.0,
    "go_nogo_cue_delay_s": 1.0,
    "burst_delay_after_go_nogo_cue_s": 0.25,

    # MOTOR-UNIT POPULATION
    "nb_motoneurons": 40,
    "min_soma_diameter": 50,
    "max_soma_diameter": 100,
    "tau_size_mode": "tau_size_um",
    "tau_size_ratio": 0.2,
    "tau_size_um": 10.0,

    # FORCE OUTPUT
    "enable_force_model": False,
    "save_per_mu_forces": False,
    "force_kernel_duration_factor": 6.0,
    "tetanic_force_at_min_diam": 1.0,
    "tetanic_force_at_max_diam": 10.0,
    "twitch_peak_rel_at_min_diam": 0.20,
    "twitch_peak_rel_at_max_diam": 0.80,
    "twitch_time_at_min_diam": 0.10,
    "twitch_time_at_max_diam": 0.02,
    "force_diameter_exponent": 2.0,

    # FORCE TARGET / BASELINE DRIVE
    "force_target_mode": "constant",
    "force_target_constant_percent": 30.0,
    "force_target_ramp_min_percent": 0.0,
    "force_target_ramp_max_percent": 100.0,
    "variable_excitatory_input_baseline": False,
    "excitatory_input_multiplier_of_rheobase": 2.0,
    "optimize_baseline": False,
    "opt_learning_rate": 3.0 * 1e3,
    "opt_max_iters": 100,
    "opt_gradient_smoothing_sigma_s": 0.35,

    # COMMON AND INDEPENDENT INPUTS
    "excitatory_input_baseline": 40 * 1e3,
    "lf_target_sd": 1.0 * 1e3,
    "lf_band_hz": [0.0, 5.0],
    "alpha_target_sd_final": 1.5 * 1e3,
    "alpha_band_hz": [8.0, 13.0],
    "alpha_envelope_band_hz": [0.0, 3.0],
    "beta_target_sd_final": 2.0 * 1e3,
    "beta_band_hz": [13.0, 30.0],
    "beta_envelope_band_hz": [0.0, 10.0],
    "common_input_weight_sd": 0.0,
    "common_input_weight_min": 0.0,
    "common_input_weight_max": 10.0,
    "independent_input_power": 10.0 * 1e3, # if independent_input_absolute_or_ratio == "absolute", this is in nA as sd (like the other inputs); if independent_input_absolute_or_ratio == "ratio", this is relative to common input power   
    "low_pass_filter_of_MN_independent_input": 50.0,
    "independent_input_weight_sd": 0.0,
    "independent_input_weight_max": 10.0,
    "independent_input_absolute_or_ratio": "absolute", # power expressed in nA

    # SLOW INPUT TRENDS
    "lf_trend_start_rel_go_cue_s": -1.0,
    "lf_trend_slope_nA_per_s": 0.0,
    "alpha_trend_start_rel_go_cue_s": -1.0,
    "alpha_trend_slope_nA_per_s": 0.0,
    "beta_trend_start_rel_go_cue_s": -1.0,
    "beta_trend_slope_nA_per_s": 0.0,

    # BURST / MODULATION PARAMETERS
    # The burst center is derived automatically from task timing; the public burst
    # controls are one shared sigma plus one peak amplitude per component.
    "enable_input_burst": True,
    "input_burst_sigma_ms": 100.0,
    "lf_burst_peak_nA": 0.0,
    "alpha_burst_peak_nA": 0.0,
    "beta_burst_peak_nA": 0.0,

    ### ANALYSIS PARAMETERS #####################

    # Low-pass + Band-specifiic CST baseline & modulation
    'active_unit_rate_threshold_hz': 3.0,  # Shared active-unit threshold used to decide which MUs are kept for CST, synchrony, and ISI-CV. Mean firing-rate summaries remain computed across all MUs.
    'cst_modulation_mode': 'zscore',  # 'zscore' or 'percent'; controls how alpha/beta Hilbert envelopes are expressed relative to baseline in saved outputs and summary figures.
    'scale_filter_order_to_frequency': True,  # If True, CST filter order is adapted to frequency band; affects LF/alpha/beta CST filtering only.
    'filter_order_scaling_coeff': 1.0,  # Scaling coefficient used when scale_filter_order_to_frequency=True.
    'default_freq_filter_order': 6,  # Default Butterworth order used by CST band filters when no scaling override applies.
    'lowest_freq_filter_order': 2,  # Minimum allowed CST filter order after scaling/clamping.
    'max_freq_filter_order': 10,  # Maximum allowed CST filter order after scaling/clamping.
    # Synchrony
    'sync_trace_display_mode': 'absolute',  # 'absolute' uses the raw excess-coincidence index; 'zscore' baseline-normalizes the sync trace for display.
    'enable_sync_index_analysis': True,  # Master switch for the sliding synchrony / coincidence analysis.
    'sync_bin_ms': 1.0,  # Spike-train bin size for the synchrony analysis only.
    'sync_win_ms': 100.0,  # Sliding-window duration for the synchrony analysis only.
    'sync_step_ms': 10.0,  # Step size between synchrony windows; smaller values give finer temporal resolution but cost more runtime.
    'sync_coinc_lag_ms': 6.0,  # Coincidence tolerance used when counting near-synchronous spikes in the synchrony analysis.
    'sync_direction_mode': "symmetric_mean",   # 'legacy_forward' reproduces the historical asymmetric coincidence count; 'symmetric_mean' averages forward and reverse directions.
    'sync_expectation_mode': 'analytic',  # 'analytic' uses the fast Poisson-independence expectation; 'surrogate_circular_shift' is slower and recomputes chance coincidence from circular-shift surrogates.
    'sync_n_surrogates': 100,  # Number of surrogate windows used only when sync_expectation_mode='surrogate_circular_shift'.
    'sync_surrogate_min_shift_ms': 1.0,  # Minimum circular shift for surrogate synchrony estimation; only affects sync_expectation_mode='surrogate_circular_shift'.

    # ANALYSIS / EXPORT PARAMETERS
    "active_unit_rate_threshold_hz": 3.0,
    "display_window_rel_cue_s": (-4.0, 1.0),
    "firing_rate_isi_window_rel_cue_s": (-4.0, -1.0),
    "cst_modulation_normalization_window_rel_cue_s": (-4.0, -3.0),
    "sync_analysis_window_rel_cue_s": (-4.0, 2.0),
    "sync_baseline_win_rel_cue_s": (-3.0, -2.0),
    "sync_post_win_rel_cue_s": (0.3, 0.6),
    "observation_window_defs_rel_cue_s": {
        "normalization_win": (-4.0, -3.0),
        "baseline_win": (-3.0, -2.0),
        "baseline_extended_win": (-2.0, -1.0),
        "ready_win": (-1.0, 0.0),
        "post_win": (0.0, 1.0),
    },
    "obs_baseline_window_rel_cue_s": (-3.0, -1.0),
    "modulation_spectrum_window_rel_cue_s": (-3.0, -1.0),
    "modulation_spectrum_max_freq_hz": 30.0,
    "modulation_spectrum_interval_mass_pct": 80.0,
    "feature_fitting_window_rel_cue_s": (-3.0, 1.0),
    "feature_analysis_window_rel_cue_s": (0.0, 0.5),
    "feature_buffer_s": 0.05,

    # DISPLAY / SUMMARY OVERLAYS
    "summary_window_overlay_specs": {
        "firing_rate_isi_window_rel_cue_s": {"enabled": True, "label": "FR / ISI window", "color": "#37a055", "lw": 2.0, "alpha": 1.0, "y_axes": 0.02},
        "cst_modulation_normalization_window_rel_cue_s": {"enabled": True, "label": "CST normalization window", "color": "#ffa600", "lw": 2.0, "alpha": 0.95, "y_axes": 0.05},
        "sync_baseline_win_rel_cue_s": {"enabled": True, "label": "Sync baseline window", "color": "#7e2179", "lw": 2.0, "alpha": 1.0, "y_axes": 0.08},
        "feature_fitting_window": {"enabled": True, "label": "Spline fitting window", "color": "#E09ED8", "lw": 2.0, "alpha": 1.0, "y_axes": 0.15},
        "feature_analysis_window": {"enabled": True, "label": "Feature window", "color": "#E900CA", "lw": 2.0, "alpha": 1.0, "y_axes": 0.15},
    },})

# FREE PARAMETERS DRAWN FROM SIMPLE PRIORS FOR EACH DISTINCT PARAMETER SET.
# Parameters also listed in a posterior source above are allowed here, but the
# posterior overlay will replace their sampled values before repeat expansion.
free_parameter_bounds_prior = {
    # # Baseline parameters
    # "tau_size_um": [0.0, 10.0],
    # "excitatory_input_baseline": [30_000.0, 50_000.0],
    # "lf_target_sd": [500.0, 5_000.0],
    # "alpha_target_sd_final": [0.0, 7_500.0],
    # "beta_target_sd_final": [0.0, 10_000.0],
    # "common_input_weight_sd": [0.0, 2.0],
    # "independent_input_power": [0.0, 10_000.0],
    # "independent_input_weight_sd": [0.0, 2.0],
    # # Cue-related parameters
    # "lf_trend_slope_nA_per_s": [-5_000.0, 0.0],
    # "alpha_trend_slope_nA_per_s": [-5_000.0, 5_000.0],
    # "beta_trend_slope_nA_per_s": [-5_000.0, 5_000.0],
    # # "input_burst_sigma_ms": [60.0, 150.0],
    # "lf_burst_peak_nA": [-6_000.0, 6_000.0],
    # "alpha_burst_peak_nA": [-3_000.0, 3_000.0],
    # "beta_burst_peak_nA": [-3_000.0, 3_000.0],
    # # Cue-related parameters - SPECIFIC EXAMPLE FOR FIGURE
    "lf_trend_slope_nA_per_s": [-2_000.0-1.0, -2_000.0+1.0], # [-1_500.0-1.0, -1_500.0+1.0],
    "alpha_trend_slope_nA_per_s": [4_000.0-1.0, 4_000.0+1.0],
    "beta_trend_slope_nA_per_s": [4_000.0-1.0, 4_000.0+1.0],
    # "input_burst_sigma_ms": [60.0, 150.0],
    "lf_burst_peak_nA": [-3_000.0-1.0, -3_000.0+1.0],
    "alpha_burst_peak_nA": [3_000.0-1.0, 3_000.0+1.0],
    "beta_burst_peak_nA": [3_000.0-1.0, 3_000.0+1.0],
}

# Optional exponential sampling for selected free parameters.
# key = parameter name, value = exponential rate lambda used inside the bounded sampler.
sample_with_exponential = {
    # "alpha_burst_peak_nA": 4.0,
}

# OPTIONAL FIXED GRIDS / SPECIFIED VALUES.
# These define deterministic parameter-set blocks before posterior overlay.
# Parameters listed here must not also be listed in POSTERIOR_PRIOR_SOURCES[*]['derived_parameters'].
sample_at_specified_values = {
    # "input_burst_sigma_ms": [80.0, 120.0, 160.0],
}

# NUMBER OF DISTINCT PARAMETER SETS BEFORE REPEAT EXPANSION.
nb_of_parameter_sets_from_prior = 1 # 50 # 100 # 15 = number of participants
repeats_per_parameter_set = 52 # 32 # 52 # = number of trials per participant
batch_repeat_seed = 12345
sim_parallel_cpus = 16


In [ ]:

# # # OPTIONAL POSTERIOR-DERIVED PRIOR OVERLAY HELPERS
posterior_overlay_info = {
    "enabled": False,
    "posterior_sampling_mode": None,
    "posterior_prior_sources_config": [],
    "source_names": [],
    "applied_parameters": [],
    "overridden_free_prior_parameters": [],
    "sources": [],
    "mode_sources": [],
    "mode_parameter_set_folder_name": None,
    "mode_parameter_set_label": None,
}
posterior_sampled_rows_by_source = {}


def _sanitize_posterior_mode_label(label: str) -> str:
    sanitized = re.sub(r"[^A-Za-z0-9_-]+", "_", str(label)).strip("_")
    return sanitized or "posterior_mode"


def _resolve_posterior_column_for_parameter(posterior_df: pd.DataFrame, parameter_name: str) -> str:
    if parameter_name in posterior_df.columns:
        return parameter_name
    suffix_matches = [
        col for col in posterior_df.columns
        if str(col).endswith(f"_{parameter_name}") or str(col) == parameter_name
    ]
    if len(suffix_matches) == 1:
        return suffix_matches[0]
    if len(suffix_matches) > 1:
        raise ValueError(
            f"Multiple posterior columns match parameter {parameter_name!r}: {suffix_matches}"
        )
    raise KeyError(
        f"Parameter {parameter_name!r} not found in posterior source. "
        f"Expected either an exact column or one exported SIM_PARAM_* column ending with _{parameter_name}."
    )


def _load_filtered_posterior_rows(
    posterior_path: Path,
    observation_label: str,
    *,
    source_kind: str,
) -> pd.DataFrame:
    posterior_path = Path(posterior_path)
    if not posterior_path.exists():
        raise FileNotFoundError(f"Posterior source file not found: {posterior_path}")
    posterior_df = pd.read_pickle(posterior_path)
    if not isinstance(posterior_df, pd.DataFrame):
        raise TypeError(f"Unexpected posterior source payload type: {type(posterior_df)}")
    if "observation_label" not in posterior_df.columns:
        raise KeyError(f"{posterior_path.name} is missing the 'observation_label' column")
    filtered = posterior_df.loc[
        posterior_df["observation_label"].astype(str) == str(observation_label)
    ].copy()
    if filtered.empty:
        available = sorted(posterior_df["observation_label"].dropna().astype(str).unique().tolist())
        raise ValueError(
            f"No posterior rows found for observation_label={observation_label!r}. "
            f"Available labels: {available}"
        )
    filtered.reset_index(drop=True, inplace=True)
    if source_kind == "posterior_modes" and len(filtered) != 1:
        raise ValueError(
            f"Expected exactly one saved posterior mode row for observation_label={observation_label!r}, "
            f"found {len(filtered)} in {posterior_path.name}."
        )
    return filtered


def _infer_posterior_modes_path(posterior_samples_path: Path | None) -> Path | None:
    if posterior_samples_path is None:
        return None
    posterior_samples_path = Path(posterior_samples_path)
    if posterior_samples_path.name == "posterior_samples.pkl":
        return posterior_samples_path.with_name("posterior_modes.pkl")
    return posterior_samples_path.parent / "posterior_modes.pkl"


def _normalize_posterior_prior_sources(use_posterior_derived_priors: bool, posterior_prior_sources: list[dict] | None) -> list[dict]:
    if not use_posterior_derived_priors:
        return []
    normalized = []
    seen_names = set()
    for idx, source in enumerate(posterior_prior_sources or []):
        if not source:
            continue
        source_name = str(source.get("name") or f"posterior_source_{idx:02d}").strip()
        if not source_name:
            source_name = f"posterior_source_{idx:02d}"
        if source_name in seen_names:
            raise ValueError(f"Duplicate posterior source name: {source_name!r}")
        seen_names.add(source_name)
        source_kind = str(source.get("source_kind", "posterior_samples")).strip().lower()
        if source_kind not in {"posterior_samples", "posterior_modes"}:
            raise ValueError(
                f"Unsupported source_kind {source_kind!r} for posterior source {source_name!r}. "
                "Expected 'posterior_samples' or 'posterior_modes'."
            )
        posterior_path = source.get("posterior_path")
        posterior_samples_path = source.get("posterior_samples_path")
        posterior_modes_path = source.get("posterior_modes_path")
        if posterior_path is not None:
            if source_kind == "posterior_modes" and posterior_modes_path is None:
                posterior_modes_path = posterior_path
            elif source_kind == "posterior_samples" and posterior_samples_path is None:
                posterior_samples_path = posterior_path
        if posterior_samples_path is not None:
            posterior_samples_path = Path(posterior_samples_path)
        if posterior_modes_path is not None:
            posterior_modes_path = Path(posterior_modes_path)
        else:
            posterior_modes_path = _infer_posterior_modes_path(posterior_samples_path)
        active_path = posterior_modes_path if source_kind == "posterior_modes" else posterior_samples_path
        if active_path is None:
            raise ValueError(f"Posterior source {source_name!r} is missing a posterior path.")
        derived_parameters = [
            str(p).strip() for p in (source.get("derived_parameters") or [])
            if str(p).strip()
        ]
        normalized.append(
            {
                "name": source_name,
                "source_kind": source_kind,
                "posterior_path": Path(active_path),
                "posterior_samples_path": posterior_samples_path,
                "posterior_modes_path": posterior_modes_path,
                "observation_label": str(source.get("observation_label")),
                "derived_parameters": derived_parameters,
                "random_seed": source.get("random_seed"),
            }
        )
    return normalized


def _numeric_posterior_columns(posterior_df: pd.DataFrame, resolved_columns: dict[str, str], *, source_name: str) -> pd.DataFrame:
    numeric_df = posterior_df.copy()
    for raw_name, posterior_col in resolved_columns.items():
        numeric = pd.to_numeric(numeric_df[posterior_col], errors="coerce")
        if numeric.isna().all():
            raise ValueError(
                f"Posterior column {posterior_col!r} for parameter {raw_name!r} in source {source_name!r} "
                "contains no usable numeric values."
            )
        numeric_df[posterior_col] = numeric
    return numeric_df


def _density_mode_row_from_samples(
    posterior_df: pd.DataFrame,
    resolved_columns: dict[str, str],
    *,
    source_name: str,
    knn_neighbors: int,
    min_neighbors: int,
) -> tuple[pd.Series, int, float]:
    numeric_df = _numeric_posterior_columns(posterior_df, resolved_columns, source_name=source_name)
    columns = [resolved_columns[p] for p in resolved_columns]
    sample_matrix = numeric_df[columns].to_numpy(dtype=float)
    finite_mask = np.all(np.isfinite(sample_matrix), axis=1)
    if not np.any(finite_mask):
        raise ValueError(f"No finite posterior samples are available for mode fallback in source {source_name!r}.")
    finite_df = numeric_df.loc[finite_mask].reset_index(drop=True)
    finite_samples = sample_matrix[finite_mask]
    if finite_samples.shape[0] == 1:
        return finite_df.iloc[0].copy(), 0, float("inf")
    means = np.nanmean(finite_samples, axis=0)
    stds = np.nanstd(finite_samples, axis=0)
    stds = np.where(np.isfinite(stds) & (stds > 0.0), stds, 1.0)
    standardized = (finite_samples - means) / stds
    diffs = standardized[:, None, :] - standardized[None, :, :]
    distances = np.sqrt(np.sum(diffs * diffs, axis=2))
    np.fill_diagonal(distances, np.inf)
    max_k = max(1, standardized.shape[0] - 1)
    k = min(max_k, max(1, int(min_neighbors), min(int(knn_neighbors), max_k)))
    nearest = np.partition(distances, kth=k - 1, axis=1)[:, :k]
    mean_knn_distance = np.nanmean(nearest, axis=1)
    density_scores = 1.0 / np.maximum(mean_knn_distance, 1e-12)
    best_index = int(np.nanargmax(density_scores))
    return finite_df.iloc[best_index].copy(), best_index, float(density_scores[best_index])


def _resolve_posterior_mode_for_source(
    source: dict,
    *,
    fallback_to_samples: bool,
    knn_neighbors: int,
    min_neighbors: int,
) -> tuple[pd.Series, dict[str, str], dict, pd.DataFrame]:
    source_name = source["name"]
    derived_parameters = source["derived_parameters"]
    if not derived_parameters:
        raise ValueError(f"Posterior mode source {source_name!r} has no derived parameters.")

    mode_error = None
    mode_path = source.get("posterior_modes_path")
    mode_df = None
    if mode_path is not None:
        try:
            mode_df = _load_filtered_posterior_rows(
                mode_path,
                source["observation_label"],
                source_kind="posterior_modes",
            )
        except (FileNotFoundError, ValueError) as exc:
            mode_error = exc

    if mode_df is not None:
        resolved_columns = {
            raw_name: _resolve_posterior_column_for_parameter(mode_df, raw_name)
            for raw_name in derived_parameters
        }
        mode_df = _numeric_posterior_columns(mode_df, resolved_columns, source_name=source_name)
        metadata = {
            "mode_resolution": "saved_posterior_modes",
            "posterior_modes_path": str(mode_path),
            "posterior_samples_path": None if source.get("posterior_samples_path") is None else str(source["posterior_samples_path"]),
            "mode_row_index": 0,
            "posterior_density_score": mode_df.iloc[0].get("posterior_density_score", np.nan),
            "fallback_warning": None,
        }
        return mode_df.iloc[0].copy(), resolved_columns, metadata, mode_df.iloc[[0]].copy()

    if not fallback_to_samples:
        raise FileNotFoundError(
            f"No usable saved posterior mode for source {source_name!r}, observation_label={source['observation_label']!r}. "
            f"Expected mode path: {mode_path}. Original error: {mode_error}"
        )
    samples_path = source.get("posterior_samples_path")
    if samples_path is None:
        raise FileNotFoundError(
            f"Cannot recompute posterior mode for source {source_name!r}: no posterior_samples_path is available. "
            f"Expected mode path: {mode_path}. Original error: {mode_error}"
        )
    warning_message = (
        f"No usable saved posterior mode for source {source_name!r} at {mode_path}; "
        f"recomputing a density-based mode from posterior samples at {samples_path}."
    )
    warnings.warn(warning_message, RuntimeWarning, stacklevel=2)
    sample_df = _load_filtered_posterior_rows(
        samples_path,
        source["observation_label"],
        source_kind="posterior_samples",
    )
    resolved_columns = {
        raw_name: _resolve_posterior_column_for_parameter(sample_df, raw_name)
        for raw_name in derived_parameters
    }
    mode_row, mode_index, density_score = _density_mode_row_from_samples(
        sample_df,
        resolved_columns,
        source_name=source_name,
        knn_neighbors=knn_neighbors,
        min_neighbors=min_neighbors,
    )
    metadata = {
        "mode_resolution": "density_fallback_from_samples",
        "posterior_modes_path": None if mode_path is None else str(mode_path),
        "posterior_samples_path": str(samples_path),
        "mode_row_index": int(mode_index),
        "posterior_density_score": float(density_score),
        "fallback_warning": warning_message,
    }
    return mode_row, resolved_columns, metadata, mode_row.to_frame().T.copy()


def _default_composite_mode_label(normalized_sources: list[dict]) -> str:
    parts = [
        f"{_sanitize_posterior_mode_label(src['name'])}-{_sanitize_posterior_mode_label(src['observation_label'])}"
        for src in normalized_sources
        if src.get("derived_parameters")
    ]
    return "__".join(parts) or "posterior_mode"


def _append_mode_rows_by_source(
    sampled_rows_by_source: dict[str, pd.DataFrame],
    source_name: str,
    mode_df: pd.DataFrame,
) -> None:
    if source_name in sampled_rows_by_source and not sampled_rows_by_source[source_name].empty:
        sampled_rows_by_source[source_name] = pd.concat(
            [sampled_rows_by_source[source_name], mode_df],
            ignore_index=True,
            sort=False,
        )
    else:
        sampled_rows_by_source[source_name] = mode_df


def overlay_posterior_derived_priors(
    parameter_sets_df: pd.DataFrame,
    *,
    use_posterior_derived_priors: bool,
    posterior_prior_sources: list[dict] | None,
    free_parameter_bounds_prior: dict,
    sample_at_specified_values: dict,
    add_posterior_mode_parameter_set: bool = False,
    posterior_mode_parameter_set_label: str | None = None,
    posterior_mode_fallback_to_samples: bool = True,
    posterior_mode_knn_neighbors: int = 25,
    posterior_mode_min_neighbors: int = 5,
) -> tuple[pd.DataFrame, dict, dict[str, pd.DataFrame]]:
    normalized_sources = _normalize_posterior_prior_sources(use_posterior_derived_priors, posterior_prior_sources)
    info = {
        "enabled": bool(use_posterior_derived_priors),
        "posterior_sampling_mode": "joint_rows" if use_posterior_derived_priors else None,
        "posterior_prior_sources_config": normalized_sources,
        "source_names": [src["name"] for src in normalized_sources],
        "applied_parameters": [],
        "overridden_free_prior_parameters": [],
        "sources": [],
        "mode_sources": [],
        "mode_parameter_set_folder_name": None,
        "mode_parameter_set_label": None,
    }
    if not use_posterior_derived_priors:
        print("[info] Posterior-derived priors disabled.")
        return parameter_sets_df.copy(), info, {}
    if not normalized_sources:
        print("[info] No posterior prior sources listed; returning dataframe unchanged.")
        return parameter_sets_df.copy(), info, {}

    specified_parameters = set(sample_at_specified_values.keys())
    parameter_owners = {}
    for source in normalized_sources:
        for parameter_name in source["derived_parameters"]:
            if parameter_name in specified_parameters:
                raise ValueError(
                    f"Parameter {parameter_name!r} is both posterior-derived and listed in sample_at_specified_values. "
                    "Remove it from one source of truth."
                )
            if parameter_name in parameter_owners:
                raise ValueError(
                    "Posterior-derived parameters must be owned by exactly one posterior source. "
                    f"Parameter {parameter_name!r} appears in both {parameter_owners[parameter_name]!r} and {source['name']!r}."
                )
            parameter_owners[parameter_name] = source["name"]

    sample_sources = [src for src in normalized_sources if src["source_kind"] == "posterior_samples"]
    out_df = parameter_sets_df.reset_index(drop=True).copy()
    sampled_rows_by_source = {}
    info["applied_parameters"] = sorted(parameter_owners.keys())
    info["overridden_free_prior_parameters"] = sorted(
        set(parameter_owners.keys()).intersection(free_parameter_bounds_prior.keys())
    )

    n_parameter_sets = len(out_df)
    for source in sample_sources:
        source_name = source["name"]
        derived_parameters = source["derived_parameters"]
        if not derived_parameters:
            print(f"[info] Source {source_name!r} has no derived parameters; skipping.")
            continue
        if source.get("posterior_samples_path") is None:
            raise ValueError(f"Posterior sample source {source_name!r} is missing posterior_samples_path.")

        posterior_df = _load_filtered_posterior_rows(
            source["posterior_samples_path"],
            source["observation_label"],
            source_kind="posterior_samples",
        )
        resolved_columns = {
            raw_name: _resolve_posterior_column_for_parameter(posterior_df, raw_name)
            for raw_name in derived_parameters
        }
        posterior_df = _numeric_posterior_columns(posterior_df, resolved_columns, source_name=source_name)
        rng = np.random.default_rng(source["random_seed"])
        replace = len(posterior_df) < n_parameter_sets
        sampled_indices = rng.choice(len(posterior_df), size=n_parameter_sets, replace=replace)
        sampled_rows = posterior_df.iloc[sampled_indices].reset_index(drop=True).copy()
        sampled_rows.insert(0, "posterior_source_name", source_name)
        sampled_rows.insert(1, "posterior_source_row_index", sampled_indices)
        sampled_rows["posterior_sampling_mode"] = "joint_rows"

        for raw_name, posterior_col in resolved_columns.items():
            out_df[raw_name] = sampled_rows[posterior_col].to_numpy(dtype=float)
            sampled_rows[raw_name] = sampled_rows[posterior_col].to_numpy(dtype=float)

        source_info = {
            "name": source_name,
            "source_kind": source["source_kind"],
            "posterior_path": str(source["posterior_samples_path"]),
            "posterior_samples_path": str(source["posterior_samples_path"]),
            "posterior_modes_path": None if source.get("posterior_modes_path") is None else str(source["posterior_modes_path"]),
            "posterior_observation_label": source["observation_label"],
            "posterior_derived_parameters": list(derived_parameters),
            "posterior_random_seed": source.get("random_seed"),
            "posterior_sampling_mode": "joint_rows",
            "posterior_sampling_with_replacement": bool(replace),
            "num_available_rows": int(len(posterior_df)),
            "num_sampled_rows": int(n_parameter_sets),
            "resolved_posterior_columns": dict(resolved_columns),
            "overridden_free_prior_parameters": sorted(
                set(derived_parameters).intersection(free_parameter_bounds_prior.keys())
            ),
            "posterior_parameter_set_folder_name": None,
        }
        info["sources"].append(source_info)

        provenance_cols = [
            c for c in [
                "posterior_source_name",
                "posterior_source_row_index",
                "posterior_sampling_mode",
                "observation_label",
                "DATASET_CONTEXT_participant_id",
                "DATASET_CONTEXT_session_tag",
                "DATASET_CONTEXT_condition_id",
            ] if c in sampled_rows.columns
        ]
        provenance_cols.extend([
            resolved_columns[p] for p in derived_parameters if resolved_columns[p] in sampled_rows.columns
        ])
        sampled_rows_by_source[source_name] = sampled_rows.loc[
            :, list(dict.fromkeys(provenance_cols + derived_parameters))
        ].copy()

        print(f"[info] Posterior source enabled: {source_name!r}")
        print(f"[info] source_kind = {source['source_kind']}")
        print(f"[info] posterior path = {source['posterior_samples_path']}")
        print(f"[info] observation_label = {source['observation_label']!r}")
        print(f"[info] posterior rows available = {len(posterior_df)}; sampled = {n_parameter_sets}; replacement = {replace}")
        print(f"[info] posterior-derived parameters = {derived_parameters}")

    if add_posterior_mode_parameter_set:
        if out_df.empty:
            raise ValueError("Cannot append a posterior-mode parameter set because parameter_sets_df is empty.")
        raw_mode_label = (
            str(posterior_mode_parameter_set_label)
            if posterior_mode_parameter_set_label is not None
            else _default_composite_mode_label(normalized_sources)
        )
        mode_prefix = "parameter_set_posterior_mode__"
        if raw_mode_label.startswith(mode_prefix):
            raw_mode_label = raw_mode_label[len(mode_prefix):]
        mode_label = _sanitize_posterior_mode_label(raw_mode_label)
        mode_folder_name = f"{mode_prefix}{mode_label}"
        mode_row = out_df.iloc[[0]].copy().reset_index(drop=True)
        mode_row["parameter_set_folder_name"] = mode_folder_name
        mode_row["posterior_mode_observation_label"] = mode_label
        mode_row["posterior_mode_source_names"] = "|".join(src["name"] for src in normalized_sources if src.get("derived_parameters"))
        mode_row["posterior_mode_source_labels"] = "|".join(
            f"{src['name']}:{src['observation_label']}" for src in normalized_sources if src.get("derived_parameters")
        )

        resolution_summaries = []
        for source in normalized_sources:
            source_name = source["name"]
            derived_parameters = source["derived_parameters"]
            if not derived_parameters:
                continue
            posterior_mode_row, resolved_columns, mode_metadata, source_mode_df = _resolve_posterior_mode_for_source(
                source,
                fallback_to_samples=posterior_mode_fallback_to_samples,
                knn_neighbors=posterior_mode_knn_neighbors,
                min_neighbors=posterior_mode_min_neighbors,
            )
            for raw_name, posterior_col in resolved_columns.items():
                mode_row.loc[0, raw_name] = float(posterior_mode_row[posterior_col])
            source_summary = {
                "name": source_name,
                "source_kind": source["source_kind"],
                "posterior_observation_label": source["observation_label"],
                "posterior_derived_parameters": list(derived_parameters),
                "resolved_posterior_columns": dict(resolved_columns),
                "overridden_free_prior_parameters": sorted(
                    set(derived_parameters).intersection(free_parameter_bounds_prior.keys())
                ),
                "posterior_parameter_set_folder_name": mode_folder_name,
                **mode_metadata,
            }
            info["mode_sources"].append(source_summary)
            resolution_summaries.append(
                f"{source_name}:{mode_metadata['mode_resolution']}:{source['observation_label']}"
            )

            source_mode_df = source_mode_df.copy()
            source_mode_df.insert(0, "posterior_source_name", source_name)
            source_mode_df.insert(1, "posterior_source_row_index", int(mode_metadata.get("mode_row_index", 0)))
            source_mode_df["posterior_sampling_mode"] = "composite_mode_row"
            source_mode_df["parameter_set_folder_name"] = mode_folder_name
            source_mode_df["posterior_mode_observation_label"] = mode_label
            source_mode_df["posterior_mode_source_names"] = mode_row.loc[0, "posterior_mode_source_names"]
            source_mode_df["posterior_mode_source_labels"] = mode_row.loc[0, "posterior_mode_source_labels"]
            source_mode_df["posterior_mode_resolution_summary"] = ";".join(resolution_summaries)
            for raw_name, posterior_col in resolved_columns.items():
                if raw_name not in source_mode_df.columns:
                    source_mode_df[raw_name] = source_mode_df[posterior_col]
            provenance_cols = [
                c for c in [
                    "posterior_source_name",
                    "posterior_source_row_index",
                    "posterior_sampling_mode",
                    "observation_label",
                    "parameter_set_folder_name",
                    "posterior_mode_observation_label",
                    "posterior_mode_source_names",
                    "posterior_mode_source_labels",
                    "posterior_mode_resolution_summary",
                    "DATASET_CONTEXT_participant_id",
                    "DATASET_CONTEXT_session_tag",
                    "DATASET_CONTEXT_condition_id",
                ] if c in source_mode_df.columns
            ]
            provenance_cols.extend([
                resolved_columns[p] for p in derived_parameters if resolved_columns[p] in source_mode_df.columns
            ])
            _append_mode_rows_by_source(
                sampled_rows_by_source,
                source_name,
                source_mode_df.loc[:, list(dict.fromkeys(provenance_cols + derived_parameters))].copy(),
            )

        mode_row["posterior_mode_resolution_summary"] = ";".join(resolution_summaries)
        for _source_name, _sampled_df in sampled_rows_by_source.items():
            if "parameter_set_folder_name" not in _sampled_df.columns:
                continue
            _mode_mask = _sampled_df["parameter_set_folder_name"].astype(str) == mode_folder_name
            if _mode_mask.any() and "posterior_mode_resolution_summary" in _sampled_df.columns:
                sampled_rows_by_source[_source_name].loc[
                    _mode_mask, "posterior_mode_resolution_summary"
                ] = mode_row.loc[0, "posterior_mode_resolution_summary"]
        out_df = pd.concat([out_df, mode_row], ignore_index=True)
        info["mode_parameter_set_folder_name"] = mode_folder_name
        info["mode_parameter_set_label"] = mode_label
        if sample_sources:
            info["posterior_sampling_mode"] = "joint_rows_with_composite_mode"
        else:
            info["posterior_sampling_mode"] = "composite_mode_row"

        print("[info] Composite posterior-mode parameter set enabled.")
        print(f"[info] mode parameter-set folder = {mode_folder_name}")
        print(f"[info] mode source labels = {mode_row.loc[0, 'posterior_mode_source_labels']}")
        print(f"[info] mode resolution summary = {mode_row.loc[0, 'posterior_mode_resolution_summary']}")

    uniform_only_parameters = sorted(set(free_parameter_bounds_prior.keys()) - set(parameter_owners.keys()))
    print(f"[info] uniform-prior-only parameters = {uniform_only_parameters}")
    return out_df, info, sampled_rows_by_source


In [ ]:
# # # SAMPLE FROM THE UNIFORM-PRIOR FREE PARAMETERS AND CREATE A DATA FRAME
prior_from_uniform_distrib_df = {}

def sample_truncated_exponential(low, high, exponent, size):
    """
    Draw `size` samples from the density
      p(x) = exponent * exp(-exponent*(x-low))  / (1 - exp(-exponent*(high-low)))
    for x in [low,high].
    This makes smaller x more likely, with rate=lam.
    """
    u = np.random.rand(*size)  # uniform [0,1)
    Z = 1 - np.exp(-exponent*(high - low))   # normalization denominator
    return low - (1.0/exponent)*np.log(1 - u * Z)

for free_param_name, free_param_bounds in free_parameter_bounds_prior.items():
    # print(free_param_name)
    # print(free_param_bounds)
    # print(np.ndim(free_param_bounds))
    arr = np.asarray(free_param_bounds)
    if arr.ndim == 1:
        # simple [low, high]
        low, high = arr
        if free_param_name in sample_with_exponential.keys():
            exponent = sample_with_exponential[free_param_name]
            print(f"Sampling {free_param_name} from an exponential distribution (λ={exponent})")
            random_sample_from_prior = sample_truncated_exponential(low, high, exponent,
                      size=(nb_of_parameter_sets_from_prior,))
        else:
            random_sample_from_prior = np.random.uniform(low, high, size=nb_of_parameter_sets_from_prior)
    elif arr.ndim == 2:
        # vector bounds: shape (n_dims, 2)
        lows  = arr[:, 0]   # length = n_dims
        highs = arr[:, 1]
        # draws an array (nb_sims × n_dims)
        if free_param_name in sample_with_exponential.keys():
            exponent = sample_with_exponential[free_param_name]
            print(f"Sampling {free_param_name} from an exponential distribution (λ={exponent})")
            # one exponential draw per dimension per sample
            random_sample_from_prior = np.column_stack([
                sample_truncated_exponential(l, h, exponent, size=(nb_of_parameter_sets_from_prior,))
                for l, h in zip(lows, highs)
            ])
        else:
            random_sample_from_prior = np.random.uniform(lows, highs,
                                        size=(nb_of_parameter_sets_from_prior, arr.shape[0]))
    else:
        raise ValueError(f"Can’t handle bounds of dimension {arr.ndim} for '{free_param_name}'")
    
    prior_from_uniform_distrib_df[free_param_name] = list(random_sample_from_prior)

prior_from_uniform_distrib_df = pd.DataFrame(prior_from_uniform_distrib_df,
                        index=range(nb_of_parameter_sets_from_prior))

In [ ]:
if prior_from_uniform_distrib_df.empty:
    prior_samples_df = pd.DataFrame(index=range(nb_of_parameter_sets_from_prior))
else:
    prior_samples_df = prior_from_uniform_distrib_df.copy()
prior_samples_df

In [ ]:
# Divide prior_sample_df into equal nb of rows for each combination of 'sample_at_specified_values', and assign new columns with names = keys of 'sample_at_specified_values' and values = value combinations of 'sample_at_specified_values'
# Returns an error if not at least one simulaton per combination
# Will return the max nb of parameter sets so that each combination 'sample_at_specified_values'
# has the same number of elements while making sure the number of retained rows stays <= nb_of_parameter_sets_from_prior.
def assign_specified_values(prior_samples_df: pd.DataFrame,
                            sample_at_specified_values: dict,
                            shuffle: bool = True,
                            random_state: int | None = None) -> pd.DataFrame:
    """
    Split prior_samples_df evenly across all combinations of `sample_at_specified_values`
    and add new columns (keys) filled with the corresponding combination values.

    - If `sample_at_specified_values` is empty, returns df unchanged.
    - Errors if there aren't enough rows to give >=1 row per combination.
    - Uses the maximum even allocation: sims_per_combo = floor(N / n_combos).
    - Drops leftover rows so all combinations have equal counts.
    """
    # Handle empty dict gracefully
    if not sample_at_specified_values:
        print("[info] No specified values provided; returning dataframe unchanged.")
        return prior_samples_df

    # Build combinations (preserve key order)
    items = list(sample_at_specified_values.items())
    keys  = [k for k, _ in items]
    grids = [v for _, v in items]
    combos = list(itertools.product(*grids))
    n_combos = len(combos)
    N = len(prior_samples_df)

    print(f"[info] available rows = {N}")
    print(f"[info] number of combinations = {n_combos}")

    sims_per_combo = N // n_combos
    if sims_per_combo < 1:
        raise ValueError(
            f"Not enough simulations: need >= {n_combos} rows (one per combination), "
            f"but only have {N}."
        )

    total_used = sims_per_combo * n_combos
    discarded = N - total_used
    print(f"[info] sims per combination = {sims_per_combo}")
    print(f"[info] total rows used = {total_used}"
          + (f" (discarding {discarded} extra rows to keep groups equal)" if discarded else ""))

    # Optionally shuffle to avoid bias, then keep only the needed rows
    idx = np.arange(N)
    rng = np.random.default_rng(random_state)
    if shuffle:
        rng.shuffle(idx)
    idx_used = idx[:total_used]

    df = prior_samples_df.iloc[idx_used].copy()
    df.reset_index(drop=True, inplace=True)

    def _column_kind_from_grid(vals):
        if any(isinstance(x, (list, tuple, np.ndarray)) for x in vals):
            return "object"
        if any(isinstance(x, (bool, np.bool_)) for x in vals):
            return "bool"
        if any(isinstance(x, (float, np.floating)) for x in vals):
            return "float"
        if all(isinstance(x, (int, np.integer)) and not isinstance(x, (bool, np.bool_)) for x in vals):
            return "int"
        return "object"

    # Build each assigned column as a plain Python list first. This avoids Pandas
    # trying to unpack sequence-valued sweep entries across rows/columns.
    column_kinds = {k: _column_kind_from_grid(vals) for k, vals in items}
    assigned_columns = {}
    for k, vals in items:
        assigned_columns[k] = [None] * total_used

    # Assign values in consecutive blocks per combination
    for combo_i, combo_vals in enumerate(combos):
        start = combo_i * sims_per_combo
        end   = start + sims_per_combo
        for k, v in zip(keys, combo_vals):
            kind = column_kinds[k]
            if kind == "float":
                v = float(v)
            elif kind == "int":
                v = int(v)
            elif kind == "bool":
                v = bool(v)
            for row_i in range(start, end):
                assigned_columns[k][row_i] = v

    # Materialize the columns using the type implied by the original sweep grid.
    for k, _vals in items:
        kind = column_kinds[k]
        if kind == "bool":
            df[k] = pd.Series(assigned_columns[k], dtype=bool)
        elif kind == "int":
            df[k] = pd.Series(assigned_columns[k], dtype=np.int64)
        elif kind == "float":
            df[k] = pd.Series(assigned_columns[k], dtype=float)
        else:
            df[k] = pd.Series(assigned_columns[k], dtype=object)

    print("[done] assignment complete.")
    return df

def _generate_unique_random_seeds(n, seed=None):
    rng = np.random.default_rng(seed)
    seeds = []
    seen = set()
    while len(seeds) < n:
        candidate = int(rng.integers(0, 2**32 - 1, dtype=np.uint32))
        if candidate in seen:
            continue
        seen.add(candidate)
        seeds.append(candidate)
    return seeds

def _next_parameter_set_id_start(batch_name_original: str,
                               add_to_same_batch_simulation_folder: bool) -> int:
    """
    When appending to an existing batch folder, continue numbering parameter_set_####
    after the highest existing index so reruns never merge into old parameter-set folders.
    """
    if not add_to_same_batch_simulation_folder:
        return 0
    batch_dir = Path(batch_name_original)
    if not batch_dir.exists():
        return 0
    pattern = re.compile(r"^parameter_set_(\d+)$")
    existing_ids = []
    for child in batch_dir.iterdir():
        if not child.is_dir():
            continue
        match = pattern.match(child.name)
        if match:
            existing_ids.append(int(match.group(1)))
    return (max(existing_ids) + 1) if existing_ids else 0


def expand_parameter_sets_with_repeats(parameter_sets_df: pd.DataFrame,
                                       repeats_per_parameter_set: int,
                                       random_seed_base: int | None = None,
                                       parameter_set_id_start: int = 0) -> pd.DataFrame:
    if repeats_per_parameter_set < 1:
        raise ValueError("repeats_per_parameter_set must be >= 1")
    parameter_sets_df = parameter_sets_df.reset_index(drop=True).copy()
    parameter_sets_df["parameter_set_id"] = np.arange(
        int(parameter_set_id_start),
        int(parameter_set_id_start) + len(parameter_sets_df),
        dtype=int,
    )
    repeated = parameter_sets_df.loc[parameter_sets_df.index.repeat(repeats_per_parameter_set)].copy()
    repeated.reset_index(drop=True, inplace=True)
    repeated["repeat_index"] = np.tile(np.arange(repeats_per_parameter_set, dtype=int), len(parameter_sets_df))
    if "parameter_set_folder_name" in repeated.columns:
        repeated.loc[repeated["parameter_set_folder_name"].notna(), "parameter_set_id"] = -1
    repeated["requested_random_seed"] = _generate_unique_random_seeds(len(repeated), seed=random_seed_base)
    return repeated

parameter_set_id_start = _next_parameter_set_id_start(
    batch_name_original=batch_name_original,
    add_to_same_batch_simulation_folder=add_to_same_batch_simulation_folder,
)
print(f"Parameter-set numbering starts at: {parameter_set_id_start}")

parameter_sets_df = assign_specified_values(prior_samples_df, sample_at_specified_values,
                                      shuffle=True)
parameter_sets_df, posterior_overlay_info, posterior_sampled_rows_by_source = overlay_posterior_derived_priors(
    parameter_sets_df,
    use_posterior_derived_priors=USE_POSTERIOR_DERIVED_PRIORS,
    posterior_prior_sources=POSTERIOR_PRIOR_SOURCES,
    free_parameter_bounds_prior=free_parameter_bounds_prior,
    sample_at_specified_values=sample_at_specified_values,
    add_posterior_mode_parameter_set=ADD_POSTERIOR_MODE_PARAMETER_SET,
    posterior_mode_parameter_set_label=POSTERIOR_MODE_PARAMETER_SET_LABEL,
    posterior_mode_fallback_to_samples=POSTERIOR_MODE_FALLBACK_TO_SAMPLES,
    posterior_mode_knn_neighbors=POSTERIOR_MODE_KNN_NEIGHBORS,
    posterior_mode_min_neighbors=POSTERIOR_MODE_MIN_NEIGHBORS,
)

prior_samples_df = expand_parameter_sets_with_repeats(
    parameter_sets_df,
    repeats_per_parameter_set=repeats_per_parameter_set,
    random_seed_base=batch_repeat_seed,
    parameter_set_id_start=parameter_set_id_start,
)

# Quick sanity check
print("Final parameter-set rows before repeats:", len(parameter_sets_df))
print("Final total rows after repeats:", len(prior_samples_df))
if USE_POSTERIOR_DERIVED_PRIORS and posterior_sampled_rows_by_source:
    for _source_name, _sampled_rows_df in posterior_sampled_rows_by_source.items():
        if _sampled_rows_df.empty:
            continue
        print(f"Posterior-derived row preview for source {_source_name!r}:")
        print(_sampled_rows_df.head().to_string(index=False))
print("Parameter-set preview:")
parameter_sets_df.head()


In [ ]:
# # # Checking which upper bound should be used for the excitatory baseline
# posterior_df_temp = posterior_df[posterior_df['muscle']=='VL']
# posterior_df_temp = posterior_df_temp[posterior_df_temp['intensity']==40]
# np.max(posterior_df_temp.index)

In [ ]:
# 1) A mapping of input names to functions that insert Brian2 units when needed.
_init_false_field_aliases = {
    # Compatibility: if an old dataframe row carries the derived dataclass field,
    # treat it as an explicit requested seed unless that was already provided.
    'random_seed': 'requested_random_seed',
}

_ignored_derived_fields = {
    'duration_with_ignored_window',
    'total_nb_motoneurons',
    'task_event_times_s',
    'input_burst_center_ms',
}

_ignored_meta_fields = {
    'parameter_set_folder_name',
    'posterior_mode_observation_label',
    'posterior_mode_source_names',
    'posterior_mode_source_labels',
    'posterior_mode_resolution_summary',
}

def _coerce_conductance_value(val):
    if isinstance(val, Quantity):
        return val
    return float(val) * msiemens


_special_setters = {
    # Backward-compatible alias kept in case an old notebook cell still uses the former name.
    'AHP_conductance_delta_after_spiking': lambda v, kw, default: kw.update({
         'AHP_fast_conductance_delta_after_spiking': _coerce_conductance_value(v)}),
    'AHP_fast_conductance_delta_after_spiking': lambda v, kw, default: kw.update({
         'AHP_fast_conductance_delta_after_spiking': _coerce_conductance_value(v)}),
    'AHP_slow_conductance_delta_after_spiking': lambda v, kw, default: kw.update({
         'AHP_slow_conductance_delta_after_spiking': _coerce_conductance_value(v)}),
}


def _coerce_like_default(name, val, default):
    """Cast values using the dataclass field type first, then fall back to the default value."""
    field_map = {f.name: f for f in fields(SimulationParameters)}
    field_info = field_map.get(name)
    annotation = None if field_info is None else field_info.type

    if annotation is float:
        return float(val)

    if annotation is int:
        if isinstance(val, (float, np.floating)):
            if not float(val).is_integer():
                raise TypeError(f"'{name}' expects an int-like value, got non-integer float {val}")
            return int(val)
        return int(val)

    if not hasattr(default, name):
        return val
    exemplar = getattr(default, name)

    if isinstance(exemplar, (int, np.integer)) and not isinstance(exemplar, bool):
        if isinstance(val, (float, np.floating)):
            if not float(val).is_integer():
                raise TypeError(f"'{name}' expects an int-like value, got non-integer float {val}")
            return int(val)
        return int(val)

    if isinstance(exemplar, Quantity):
        if isinstance(val, Quantity):
            return val
        scalar = np.asarray(exemplar, dtype=float)
        if scalar.size == 1 and float(scalar) != 0.0:
            exemplar_unit = exemplar / float(scalar)
            return float(val) * exemplar_unit
        return val

    if isinstance(exemplar, np.ndarray):
        try:
            return np.asarray(val, dtype=exemplar.dtype)
        except Exception:
            return np.asarray(val)

    if isinstance(exemplar, (float, np.floating)):
        return float(val)

    return val


def build_simulation_parameters(fixed_parameters, free_priors, batch_name):
    default = SimulationParameters()
    custom_folder_name = free_priors.get("parameter_set_folder_name", fixed_parameters.get("parameter_set_folder_name"))
    parameter_set_id = free_priors.get("parameter_set_id", fixed_parameters.get("parameter_set_id", -1))
    if isinstance(custom_folder_name, str) and custom_folder_name.strip():
        output_folder_name = os.path.join(batch_name, custom_folder_name.strip())
    elif parameter_set_id is not None and int(parameter_set_id) >= 0:
        output_folder_name = os.path.join(batch_name, f"parameter_set_{int(parameter_set_id):04d}")
    else:
        output_folder_name = batch_name
    init_kwargs = {'output_folder_name': output_folder_name}
    dataclass_fields = {f.name for f in fields(SimulationParameters) if f.init}

    all_priors = {**fixed_parameters, **free_priors}

    for name, val in all_priors.items():
        if name in _special_setters:
            _special_setters[name](val, init_kwargs, default)
            continue

        if name in dataclass_fields:
            init_kwargs[name] = _coerce_like_default(name, val, default)
            continue

        alias = _init_false_field_aliases.get(name)
        if alias is not None:
            if alias not in all_priors and alias not in init_kwargs:
                init_kwargs[alias] = _coerce_like_default(alias, val, default)
            continue

        if name in _ignored_derived_fields or name in _ignored_meta_fields:
            continue

        raise KeyError(f"Unrecognized parameter '{name}'")

    return SimulationParameters(**init_kwargs)


In [ ]:
# # # CREATE FOLDER FOR THE SIMULATION BATCH
if add_to_same_batch_simulation_folder: # Do not create a new folder - just add the new simulations to the existing one with the same name
    batch_name = batch_name_original
    os.makedirs(batch_name, exist_ok=True)
    print(f"Using batch folder: {batch_name}")
else:
    # figure out the next batch index - if creating a new folder
    # find all subdirectories matching "{batch_name_original}_{digits}"
    pattern = re.compile(rf"^{re.escape(batch_name_original)}_(\d+)$")
    batch_dirs = [
        d for d in os.listdir('.')
        if os.path.isdir(d) and pattern.match(d)
    ]

    if batch_dirs:
        # extract the numeric suffixes
        existing_idxs = [int(pattern.match(d).group(1)) for d in batch_dirs]
        batch_idx = max(existing_idxs) + 1
    else:
        batch_idx = 0

    batch_name = f"{batch_name_original}_{batch_idx}"
    os.makedirs(batch_name, exist_ok=True)

    print(f"Created batch folder: {batch_name}")

In [ ]:
params_prior_list = []
for i in range(len(prior_samples_df)):
    free_row = prior_samples_df.iloc[i].to_dict()
    # remove elements which are not parameters - namely the subject/intensity/muscle the parameters sampled from posteriors are extracted from
    if "subject" in free_row:
        free_row.pop("subject")
    if "intensity" in free_row:
        free_row.pop("intensity")
    if "muscle" in free_row:
        free_row.pop("muscle")
    sim_param = build_simulation_parameters(fixed_parameters, free_row, batch_name)
    params_prior_list.append(sim_param)

In [ ]:
# Import for parallelization
from joblib import Parallel, delayed
from brian2 import prefs, device
import logging

# # # Function to make sure to terminate any Python process that runs in the background (this can happen when the kernel crashes during the parallelized computations)
def kill_other_python_processes():
    me = os.getpid()
    user = getpass.getuser()
    for proc in psutil.process_iter(['pid', 'name', 'username']):
        try:
            # only consider Python executables run by this user
            if proc.info['username'] != user:
                continue
            name = proc.info['name'].lower()
            # match python, pythonw, python3, etc
            if name.startswith('python'):
                pid = proc.info['pid']
                if pid != me:
                    proc.kill()   # or proc.terminate()
                    print(f"{name} process terminated")
        except (psutil.NoSuchProcess, psutil.AccessDenied):
            pass
if __name__ == '__main__':
    kill_other_python_processes()
    # now safe to start joblib Parallel(...)

# --- Helper to wrap a single simulation and then reset Brian2 ---
def _run_and_reset(params):
    # params must be a SimulationParameters instance
    out = run_simulation(params)
    # after each run, reset Brian2’s magic network so the next worker starts fresh
    device.reinit()     # clears all Brian2 objects
    device.activate()   # re–activate the default runtime device
    return out

# # # PARALLEL SIMULATIONS
def parallel_simulate(params_prior_list, n_jobs=8):
    """
    params_prior_list : list of SimulationParameters
    n_jobs      :       number of parallel workers
    """
    # Note: `prefer="processes"` is the default for `n_jobs>1`
    sim_outputs = Parallel(n_jobs=n_jobs)(
        delayed(_run_and_reset)(p) for p in params_prior_list)
    return sim_outputs

In [ ]:
# # # SAVE ALL THE PRIORS AND SAMPLED PRIOR PARAMETERS FOR THE GIVEN BATCH
# save all the prior info into a pickle
to_save = {
    'fixed_parameters':         fixed_parameters,
    'free_parameter_bounds_prior':    free_parameter_bounds_prior,
    'sample_at_specified_values':    sample_at_specified_values,
    'torch_seed':               torch.initial_seed(),  # or whatever seed you used
    'num_parameter_sets':       nb_of_parameter_sets_from_prior,
    'repeats_per_parameter_set': repeats_per_parameter_set,
    'batch_repeat_seed':        batch_repeat_seed,
    'parameter_sets_df':        parameter_sets_df,
    'prior_samples_df':         prior_samples_df,
    'params_prior_list':        params_prior_list,
    'posterior_prior_config':   posterior_overlay_info,
    'posterior_prior_sources_config': POSTERIOR_PRIOR_SOURCES,
    'posterior_sampled_rows_by_source': posterior_sampled_rows_by_source,
    'posterior_sampling_mode':  posterior_overlay_info.get('posterior_sampling_mode'),
    'posterior_sampling_with_replacement_by_source': {
        src.get('name'): src.get('posterior_sampling_with_replacement')
        for src in posterior_overlay_info.get('sources', [])
    },
}
# pkl_path = os.path.join(batch_name, f"{batch_name}_priors.pkl")
# with open(pkl_path, 'wb') as f:
#     pickle.dump(to_save, f)

# print(f"Saved priors to {pkl_path}")

# Prepare the base filename
base = f"{batch_name}_priors"
if "\\" in base:
    base = base.split("\\")[-1]

# Find existing files of the form "{base}_batch<idx>.pkl"
pattern = re.compile(rf"^{re.escape(base)}_batch(\d+)\.pkl$")
existing_idxs = []
for fn in os.listdir(batch_name):
    m = pattern.match(fn)
    if m:
        existing_idxs.append(int(m.group(1)))

# Choose the next index
if existing_idxs:
    next_idx = max(existing_idxs) + 1
else:
    next_idx = 0

# Build the new filename
filename = f"{base}_batch{next_idx}.pkl"
pkl_path = os.path.join(batch_name, filename)

# Save
with open(pkl_path, 'wb') as f:
    pickle.dump(to_save, f)

print(f"Saved priors to {pkl_path}")

# Ensure each SimulationParameters points to its parameter-set folder.
for p, (_, row) in zip(params_prior_list, prior_samples_df.iterrows()):
    custom_folder_name = row.get("parameter_set_folder_name")
    if isinstance(custom_folder_name, str) and custom_folder_name.strip():
        p.output_folder_name = os.path.join(batch_name, custom_folder_name.strip())
    elif getattr(p, "parameter_set_id", -1) >= 0:
        p.output_folder_name = os.path.join(batch_name, f"parameter_set_{int(p.parameter_set_id):04d}")
    else:
        p.output_folder_name = batch_name


In [ ]:
# Capture progres (in .log file) directly in the notebook cell
from threading import Thread, Event
import time
from datetime import datetime
import re

_tail_thread = None
_tail_stop = threading.Event()

def start_tail(logfile="simulations_progress_log.log", poll_interval=1.0):
    """
    Spawn a thread that prints only the log‐lines whose timestamp
    is ≥ the moment you called start_tail(), and strips off everything
    before the log‐level (INFO:, WARNING:, ERROR:, etc.).
    """
    global _tail_thread, _tail_stop

    # make sure the file exists (touch it)
    open(logfile, "a").close()

    # remember "now" and clear any previous stop flag
    start_dt = datetime.now()
    _tail_stop.clear()

    def _tail_loop():
        level_re = re.compile(r'\b(?:DEBUG|INFO|WARNING|ERROR|CRITICAL)\b:\s*')
        with open(logfile, "r", encoding="utf-8") as f:
            # seek to end: we only want new lines
            f.seek(0, 2)
            while not _tail_stop.is_set():
                line = f.readline()
                if not line:
                    time.sleep(poll_interval)
                    continue

                # try to parse timestamp at the very start
                try:
                    ts_str = " ".join(line.split(" ")[:2]).rstrip(",")
                    ts = datetime.strptime(ts_str, "%Y-%m-%d %H:%M:%S,%f")
                except Exception:
                    ts = start_dt  # force print for non‐timestamped lines

                if ts >= start_dt:
                    # strip off everything before the level marker
                    m = level_re.search(line)
                    if m:
                        print(line[m.start():], end="")
                    else:
                        print(line, end="")

    # fire up the thread (only one at a time)
    if _tail_thread is None or not _tail_thread.is_alive():
        _tail_thread = threading.Thread(target=_tail_loop, daemon=True)
        _tail_thread.start()
    else:
        print("Tail already running; call `end_tail()` first if you want to restart.")

def stop_tail():
    """Stop the background tail thread."""
    _tail_stop.set()
    if _tail_thread:
        _tail_thread.join()


In [ ]:
# Run parallel simulations
start_tail()
simulation_output_files = parallel_simulate(params_prior_list, n_jobs=sim_parallel_cpus)
time.sleep(1)  # give it a moment to print the last lines
stop_tail()

In [ ]:
# UNIFIED ANALYSIS / SUMMARY / EXPORT CONFIG
SIM_BATCH_SUMMARY_ROOT = Path(batch_name)
DISPLAY_XLIM_REL_CUE_S = (-4.0, 1.0)

OBSERVATION_WINDOW_CONFIG = dict(fixed_parameters['observation_window_defs_rel_cue_s'])
OBS_BASELINE_WINDOW_REL_CUE_S = tuple(fixed_parameters['obs_baseline_window_rel_cue_s'])
MODULATION_SPECTRUM_CONFIG = {
    'window_rel_cue_s': tuple(fixed_parameters['modulation_spectrum_window_rel_cue_s']),
    'max_freq_hz': float(fixed_parameters['modulation_spectrum_max_freq_hz']),
    'interval_mass_pct': float(fixed_parameters['modulation_spectrum_interval_mass_pct']),
}

FEATURE_EXTRACTION_CONFIG = {
    'enabled': True,
    'fitting_window': (-3.0, 1.0),
    'analysis_window': (0.0, 0.5),
    'buffer_s': 0.05,
    'spline_smoothing': None,
    'spline_order': 3,
    'n_knots': None,
    'trend_pre_window': (-3.0, -1.0),
    'trend_post_window': (0.0, 1.0),
    'extrapolate': False,
}

EXPORT_TRACE_SUMMARY_FIGURES = False
EXPORT_RATE_ISI_SUMMARY_FIGURES = False
SHOW_INDIVIDUAL_TRACES = False
SHOW_STD_BAND = True
SHOW_REFERENCE_STD_LINES = True
FIX_YLIM_ACROSS_PARAMETER_SETS = True
COLOR_OVERRIDES = dict(fixed_parameters['input_component_colors'])
PLOT_STYLE = {
    'figure_size': (19, 11),
    'figure_dpi': 120,
    'feature_outside_color': '#C9C9C9',
    'feature_outside_alpha': 0.20,
    'feature_baseline_color': '#111111',
    'feature_baseline_lw': 1.4,
    'feature_baseline_ls': '--',
    'feature_analysis_mean_lw': 1.4,
    'feature_analysis_mean_ls': '--',
    'feature_analysis_mean_alpha': 1.0,
}

WINDOW_OVERLAY_SPECS = dict(fixed_parameters['summary_window_overlay_specs'])


In [ ]:
sim_parameter_set_summary_df = summarize_parameter_set_batch(
    SIM_BATCH_SUMMARY_ROOT,
    show_individual_traces=SHOW_INDIVIDUAL_TRACES,
    show_std_band=SHOW_STD_BAND,
    show_reference_std_lines=SHOW_REFERENCE_STD_LINES,
    fixed_ylim_across_parameter_sets=FIX_YLIM_ACROSS_PARAMETER_SETS,
    color_overrides=COLOR_OVERRIDES,
    plot_style=PLOT_STYLE,
    xlim_rel_cue_s=DISPLAY_XLIM_REL_CUE_S,
    feature_config=FEATURE_EXTRACTION_CONFIG,
    window_overlay_specs=WINDOW_OVERLAY_SPECS,
    export_figures=(EXPORT_TRACE_SUMMARY_FIGURES and not (fixed_parameters['minimal_output'] and fixed_parameters['minimal_output_turn_off_figure_output'])),
    show_progress=True,
)

sim_rate_isi_metrics_df, sim_rate_isi_figures_df = summarize_rate_isi_batch(
    SIM_BATCH_SUMMARY_ROOT,
    export_figures=EXPORT_RATE_ISI_SUMMARY_FIGURES,
    show_progress=True,
)

sim_parameter_set_summary_df.head()


In [ ]:
sim_export_df, sim_export_paths = export_parameter_set_summary_table(
    SIM_BATCH_SUMMARY_ROOT,
    parameter_set_summary_df=sim_parameter_set_summary_df,
    rate_isi_metrics_df=sim_rate_isi_metrics_df,
    feature_config=FEATURE_EXTRACTION_CONFIG,
)

print(sim_export_paths)
sim_export_df.head()


In [ ]:
# #### MINIMUM SUMMARIZING AND EXPORT CELL, FOR RECOVERY IF THE SIMULATION WAS STOPPED BEFORE THE END OF THE BATCH

# from parameter_set_batch_summary import (
#     summarize_parameter_set_batch,
#     summarize_rate_isi_batch,
#     export_parameter_set_summary_table,
# )

# sim_parameter_set_summary_df = summarize_parameter_set_batch(
#     root_dir=BATCH_ROOT_DIR,
#     output_dir=OUTPUT_DIR,
#     show_individual_traces=SHOW_INDIVIDUAL_TRACES,
#     show_std_band=SHOW_STD_BAND,
#     show_reference_std_lines=SHOW_REFERENCE_STD_LINES,
#     fixed_ylim_across_parameter_sets=FIX_YLIM_ACROSS_PARAMETER_SETS,
#     color_overrides=COLOR_OVERRIDES,
#     plot_style=PLOT_STYLE,
#     xlim_rel_cue_s=XLIM_REL_CUE_S,
#     feature_config=FEATURE_EXTRACTION_CONFIG,
#     export_figures=EXPORT_TRACE_SUMMARY_FIGURES,
# )

# sim_rate_isi_metrics_df, sim_rate_isi_figures_df = summarize_rate_isi_batch(
#     root_dir=BATCH_ROOT_DIR,
#     output_dir=OUTPUT_DIR,
#     show_progress=True,
#     export_figures=EXPORT_RATE_ISI_SUMMARY_FIGURES,
# )

# sim_export_df, sim_export_paths = export_parameter_set_summary_table(
#     root_dir=BATCH_ROOT_DIR,
#     output_dir=OUTPUT_DIR,
#     parameter_set_summary_df=sim_parameter_set_summary_df,
#     rate_isi_metrics_df=sim_rate_isi_metrics_df,
#     feature_config=FEATURE_EXTRACTION_CONFIG,
# )
